# Starter

A working minimal Kuration SDK pipeline you can copy, run, and extend.

- Runs green out of the box against a tiny synthetic dataset.
- Ends with one TODO for you to fill in.

In Colab: `File > Save a copy in Drive`, then edit.

In [ ]:
%pip install --quiet kuration-sdk

## 1. Tiny synthetic dataset

Five devices, 500 ticks each, per-device amplitude growing with the device index. Nothing exotic, just enough to see the SDK end to end.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
N_DEVICES = 5
N_TICKS = 500

sessions = {}
for i in range(N_DEVICES):
    amplitude = 0.5 * (i + 1)
    signal = amplitude * rng.standard_normal(N_TICKS)
    sessions[f"dev_{i}"] = pd.DataFrame({
        "t": np.arange(N_TICKS),
        "device": f"dev_{i}",
        "signal": signal,
    })

print(f"Sessions: {list(sessions.keys())}")
print(f"Rows per session: {len(sessions['dev_0'])}")

## 2. A working pipeline

Read the sessions, cut them into 50-tick windows with 25-tick stride, measure the standard deviation of the signal per window, bin the results into three buckets, and sample equally across those buckets so the quiet and loud ends both show up.

In [ ]:
from kuration import input

result = (
    input(sessions, time="t", entity="device", hz=1)
    .window(50, stride=25)
    .measure("activity", "std", of="signal")
    .bin("activity", bins=3)
    .sample("equal", n=30)
    .options(seed=42)
    .collect()
)

print(f"Selected {len(result.selected)} windows.")
print("Devices in selection:", sorted({w['device'] for w in result.selected}))

## 3. Your turn (TODO)

Extend the pipeline. Some easy first steps:

- Add a second `.measure(...)` for a different feature (`"mean"`, `"min"`, `"max"`, `"rate"`).
- Try different bin counts on `.bin(...)`.
- Change the seed on `.options(seed=...)` and see how the selection shifts.
- Grow `N_DEVICES` and re-run.
- Replace the synthetic `sessions` dict with your own DataFrames.

If you get stuck, the paired notebooks in this folder show more complete pipelines: `action-state-consistency`, `collision-aware`, `coverage-based-curation`.

In [ ]:
# TODO: extend the pipeline above. Try one of the ideas from the previous cell.
# Your code here:
